In [2]:
# import packages
import pandas as pd
import numpy as np
import datetime as dt
import scipy.stats as sp
%load_ext google.cloud.bigquery
import pickle
from matplotlib import pyplot as plt
from datetime import datetime
with open('design.pkl','rb') as f:
    set_params = pickle.load(f)
plt.rcParams.update(set_params)

from google.cloud import bigquery
# Modules
from modules.connector import MyBigQuery, MySQL

The google.cloud.bigquery extension is already loaded. To reload it, use:
  %reload_ext google.cloud.bigquery


In [3]:
bq = MyBigQuery(
    credentials_file="./credentials/New AutoMotive Index-487e031dc242.json"
)

In [4]:
def ft_cleaner(x):
    x = x.replace('GAS BI-FUEL', 'Other').replace('GAS', 'Other').replace('GAS DIESEL', 'Other').replace('PETROL/GAS', 'Other').replace('LPG', 'Other').replace('STEAM', 'Other').replace('LNG', 'Other').replace('OTHER', 'Other').replace('CNG', 'Other')
    x = x.replace('FUEL CELLS', 'Other')
    x = x.replace('PETROL', 'Petrol').replace('Petrol/Other', 'Petrol')
    x = x.replace('HYBRID ELECTRIC (CLEAN)', 'Hybrid').replace('HYBRID ELECTRIC', 'Hybrid').replace('ELECTRIC DIESEL', 'Hybrid')
    x = x.replace('DIESEL', 'Diesel').replace('Other Diesel', 'Diesel')
    x = x.replace('ELECTRICITY', 'Pure Electric').replace('ELECTRIC', 'Pure Electric')
    return x

In [8]:
# co2ComplianceAdj will be the surplus after removing demand from other manufacturers
def co2ComplianceAdjFunc(row, total_trading_diff_perc):
    #checks for nan
    if row.co2Compliance > 0:
        co2ComplianceAdj = row.co2Compliance * total_trading_diff_perc
    
        return co2ComplianceAdj
    else:
        return row.co2Compliance


    
def co2_to_zev(row, mandate):
    new_zev_credits = 0
    limit = mandate*.65
    limit = limit * row.totalSales
    
    if row.co2Compliance > 0:
        new_zev_credits = row.co2ComplianceAdj / 167
    
    if new_zev_credits > limit:
        new_zev_credits = limit
        
    
    if new_zev_credits > 0:
        return new_zev_credits
    else:
        return 0

In [19]:
car_targets = bq.from_bq_to_dataframe(
query = """
SELECT manufacturer, AVG(co2Emissions) as co2Target
FROM VES_dataset.parentManufacturerM1
WHERE typeApproval = 'M1'
AND monthOfFirstRegistration > '2020-12-31'
AND monthOfFirstRegistration < '2022-01-01'
AND co2Emissions > 0
GROUP BY manufacturer

""")

In [20]:
mandate = .22

In [48]:
%%bigquery df_activity --project rugged-baton-283921
SELECT 
    VES.manufacturer,
    COUNT(*) as totalSales,
    COUNTIF(VES.co2Emissions > 0) as nonzevSales,
    COUNTIF(VES.co2Emissions = 0) as zevSales,
    SUM(VES.co2Emissions) as co2Activity

FROM VES_dataset.parentManufacturerM1 VES
WHERE typeApproval = 'M1'
AND CAST(monthOfFirstRegistration AS DATE) > '2023-12-31'
AND CAST(monthOfFirstRegistration AS DATE) < DATE_TRUNC(CURRENT_DATE(), MONTH)
# AND make IN (
#     SELECT make
#     FROM VES_dataset.VRN_UK
#     WHERE typeApproval = 'M1'
#     AND monthOfFirstRegistration > '2020-03-31'
#     AND monthOfFirstRegistration < '2023-04-01'
#     GROUP BY make
#     ORDER BY COUNT(*) DESC
#     LIMIT 20
    
# )

# Volume limit
AND manufacturer IN (
    SELECT manufacturer
    FROM VES_dataset.parentManufacturerM1
    WHERE typeApproval = 'M1'
    AND CAST(monthOfFirstRegistration AS DATE) >= DATE_SUB(DATE_TRUNC(CURRENT_DATE(), MONTH), INTERVAL 1 YEAR)
    AND CAST(monthOfFirstRegistration AS DATE) < DATE_TRUNC(CURRENT_DATE(), MONTH)
    -- AND total > 2500
    GROUP BY manufacturer
    HAVING COUNT(*) > 2500
)



GROUP BY manufacturer
ORDER BY totalSales ASC;

Downloading: 100%|██████████| 15/15 [00:01<00:00, 14.91rows/s]


In [50]:
df_activity.totalSales.sum()

131302

In [32]:
df_activity.to_csv('flourish_data/basic_activity.csv')

In [33]:
dfa = df_activity.merge(car_targets, how='left', on='manufacturer')
dfa['co2Allowances'] = dfa.apply(lambda row: row.co2Target * row.nonzevSales, axis=1)
dfa['co2Compliance'] = dfa.apply(lambda row: row.co2Allowances - row.co2Activity, axis=1)

In [34]:
# Adjust CO2 -> ZEV transfers to take into account demand from non-compliant manufacturers. 

co2_trading_demand = dfa[dfa.co2Compliance < 0].co2Compliance.sum()
co2_trading_supply = dfa[dfa.co2Compliance > 0].co2Compliance.sum()

co2_trading_diff = co2_trading_supply + co2_trading_demand
co2_trading_diff_perc = co2_trading_diff / co2_trading_supply

dfa['co2ComplianceAdj'] = dfa.apply(lambda row: co2ComplianceAdjFunc(row=row, total_trading_diff_perc = co2_trading_diff_perc), axis=1)

In [35]:
mandate = .22
dfa['ZEVMallowances'] = dfa.apply(lambda row: round((1-mandate) * row.totalSales, ndigits=2), axis=1)
dfa['ZEV Surplus'] = dfa.apply(lambda row: row.ZEVMallowances - row.nonzevSales, axis=1)

In [36]:
dfa['co2_to_mandate_allowances'] = dfa.apply(lambda row: co2_to_zev(row, mandate=mandate), axis=1)
dfa['ZEV Surplus after CO2'] = dfa.apply(lambda row: row['ZEV Surplus'] + row.co2_to_mandate_allowances, axis=1)


In [37]:
dfa.to_csv('flourish_data/full.csv')

In [47]:
dfa.rename(columns={
    'manufacturer': 'Manufacturer',
    'totalSales': 'Total sales & target',
    'nonzevSales': 'Non-ZEV sales',
    'zevSales': 'ZEV sales',
    'co2Activity': 'CO2 activity',
    'co2Target': 'CO2 target',
    'co2Allowances': 'CO2 allowances',
    'co2ComplianceAdj': 'CO2 compliance level',
    'ZEVMallowances': 'ZEV mandate allowances',
    'ZEV Surplus': 'ZEV mandate allowance surplus',
    'co2_to_mandate_allowances': 'CO2-derived ZEV mandate credits'}
          )[
['Manufacturer',
 'Total sales',
 'ZEV mandate allowances',
 'ZEV sales',
 'Non-ZEV sales',
 'CO2 activity',
 'CO2 target',
 'CO2 allowances',
 'CO2 compliance level',
 'ZEV mandate allowance surplus',
 'CO2-derived ZEV mandate credits',
 'ZEV Surplus after CO2']].to_csv('flourish_data/full.csv', index=False)